# Demo 02 · RAG explicable y evaluación con test set FraSoHome

            Esta demo construye un RAG local sobre los documentos de conocimiento de FraSoHome. Usa TF-IDF en lugar de embeddings externos para que sea reproducible sin APIs, pero mantiene la estructura conceptual de un RAG real: documentos, chunks, retrieval, respuesta con citas y evaluación.


In [1]:
from pathlib import Path
import sys
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 180)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datos").exists() and (candidate / "escenarios").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado la raíz del proyecto FraSoHome.")

PROJECT_ROOT = find_project_root()
DEMO_DIR = PROJECT_ROOT / "escenarios" / "09-genai-interpretabilidad"
ARTIFACT_DIR = DEMO_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(DEMO_DIR))

from common.genai_frasohome import (
    append_trace,
    build_agent_trace,
    build_rag_index,
    chunk_documents,
    evaluate_agent_trace,
    evaluate_rag_questions,
    grounded_answer,
    keyword_coverage,
    load_crm,
    load_knowledge_documents,
    load_sales_artifact,
    local_feature_contributions,
    recommendation_from_score,
    retrieve,
    simple_answer_rubric,
    summarize_customer_context,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEMO_DIR:", DEMO_DIR)


PROJECT_ROOT: C:\repos\frasohome-demos
DEMO_DIR: C:\repos\frasohome-demos\escenarios\09-genai-interpretabilidad


## 1. Cargar documentos y construir chunks

            La explicabilidad empieza antes del LLM. Si el documento está obsoleto, mal troceado o sin metadatos, el sistema puede fallar aunque el modelo generativo sea bueno.


In [2]:
documents = load_knowledge_documents(PROJECT_ROOT)
display(documents[["file_name", "area", "is_obsolete", "is_security_test"]])

chunks = chunk_documents(documents, max_chars=850)
print("Documentos:", len(documents))
print("Chunks:", len(chunks))
display(chunks[["chunk_id", "file_name", "area", "is_obsolete", "is_security_test"]].head(10))


,file_name,area,is_obsolete,is_security_test
0,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
1,FS-KB-02_Politica_Devoluciones_v1.2_Obsoleta.docx,01_Politicas,True,False
2,FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.docx,02_KPI_y_Datos,False,False
3,FS-KB-04_Manual_Tienda_Caja_y_Pagos_Mixtos_v2.1.docx,03_Operaciones,False,False
4,FS-KB-05_Guia_Conciliacion_Pagos_Ecommerce_v1.4.docx,03_Operaciones,False,False
5,FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx,03_Operaciones,False,False
6,FS-KB-06_Taxonomia_Catalogo_y_Reglas_SKU_v1.2.docx,04_Catalogo,False,False
7,FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx,05_CRM,False,False
8,FS-KB-08_Politica_Datos_y_Permisos_Asistente_v1.0.docx,06_Gobierno_y_Seguridad,False,False
9,FS-KB-10_Documento_Prueba_Prompt_Injection_NO_PROD.docx,07_Test_Security,False,True


Documentos: 10
Chunks: 37


,chunk_id,file_name,area,is_obsolete,is_security_test
0,FS-KB-01-CH000,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
1,FS-KB-01-CH001,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
2,FS-KB-01-CH002,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
3,FS-KB-01-CH003,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
4,FS-KB-01-CH004,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
5,FS-KB-01-CH005,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
6,FS-KB-01-CH006,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
7,FS-KB-01-CH007,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,01_Politicas,False,False
8,FS-KB-02-CH000,FS-KB-02_Politica_Devoluciones_v1.2_Obsoleta.docx,01_Politicas,True,False
9,FS-KB-02-CH001,FS-KB-02_Politica_Devoluciones_v1.2_Obsoleta.docx,01_Politicas,True,False


## 2. Construir el índice RAG

            Por defecto excluimos documentos obsoletos y documentos de prueba de seguridad. Este filtro forma parte de la explicabilidad y del gobierno del sistema.


In [3]:
rag_index = build_rag_index(chunks, include_obsolete=False, include_security_test=False)
print("Chunks indexados:", len(rag_index.chunks))
display(rag_index.chunks["file_name"].value_counts().to_frame("chunks"))


Chunks indexados: 33


,chunks
file_name,
FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,8
FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.docx,6
FS-KB-04_Manual_Tienda_Caja_y_Pagos_Mixtos_v2.1.docx,4
FS-KB-08_Politica_Datos_y_Permisos_Asistente_v1.0.docx,4
FS-KB-05_Guia_Conciliacion_Pagos_Ecommerce_v1.4.docx,3
FS-KB-06_Taxonomia_Catalogo_y_Reglas_SKU_v1.2.docx,3
FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx,3
FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx,2


## 3. Inspeccionar una consulta

            La demo muestra ranking, scores y textos. Esto permite discutir si el sistema recuperó evidencia suficiente o si el fallo está en retrieval, chunking o síntesis.


In [4]:
query = "¿Cuál es el plazo de devolución para compras online?"
retrieved = retrieve(rag_index, query, top_k=5)
display(retrieved[["rank", "score", "file_name", "chunk_id", "area"]])

answer_obj = grounded_answer(query, retrieved)
print("RESPUESTA SIMULADA:")
print(answer_obj["answer"])
print("\nCITAS:")
display(pd.DataFrame(answer_obj["citations"]))


,rank,score,file_name,chunk_id,area
21,1,0.206058,FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx,FS-KB-09-CH000,03_Operaciones
0,2,0.134957,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,FS-KB-01-CH000,01_Politicas
4,3,0.084292,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,FS-KB-01-CH004,01_Politicas
12,4,0.056887,FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.docx,FS-KB-03-CH004,02_KPI_y_Datos
5,5,0.054616,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,FS-KB-01-CH005,01_Politicas


RESPUESTA SIMULADA:
Q: ¿Cuál es el plazo de devolución online? Para devoluciones por incidencia, el plazo de reporte es de 72 horas desde la entrega/recogida (daños visibles) y 7 días para faltantes o defectos funcionales no visibles. Canal de compra | Plazo devolución voluntaria | Observaciones A: 45 días naturales desde la entrega para devoluciones voluntarias.

CITAS:


,chunk_id,file_name
0,FS-KB-09-CH000,FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx
1,FS-KB-01-CH000,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx
2,FS-KB-01-CH005,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx
3,FS-KB-09-CH000,FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx


## 4. Evaluar con el test set documental existente

            Usamos `FS_TestSet_Documental.csv`, que contiene preguntas y respuestas esperadas. La métrica de cobertura de keywords no reemplaza una revisión humana, pero ayuda a detectar regresiones y casos débiles.


In [5]:
test_path = PROJECT_ROOT / "datos" / "rag_copilot_studio" / "test_sets" / "FS_TestSet_Documental.csv"
test_set = pd.read_csv(test_path)
display(test_set)

rag_eval = evaluate_rag_questions(rag_index, test_set, top_k=5)
display(rag_eval[["question", "top_doc", "top_score", "retrieval_keyword_coverage", "answer_keyword_coverage", "citation_count"]])

summary = rag_eval[["retrieval_keyword_coverage", "answer_keyword_coverage", "citation_count"]].mean(numeric_only=True)
display(summary.to_frame("media"))


,Question,Expected response
0,¿Cuál es el plazo de devolución para compras online?,45 días naturales; devoluciones voluntarias; desde la entrega
1,¿Qué productos no admiten devolución voluntaria?,a medida; OUTLET; tarjetas regalo; servicios
2,¿Cómo se reembolsa un pago mixto tarjeta + efectivo?,primero a tarjeta; resto en efectivo
3,¿Qué significa ventas netas y cómo se calcula?,Ventas brutas - descuentos - descuento por puntos - devoluciones
4,¿Qué es stock disponible?,stock en mano - reservado - bloqueado
5,¿Qué plazo tengo para reportar un daño en transporte?,72 horas
6,¿Qué hago si el cliente no tiene ticket?,buscar por fidelización; aprobación del Store Manager
7,¿En qué casos se debe escalar una devolución por sospecha de fraude?,patrones anómalos; prevención de pérdidas
8,¿Qué información debe incluir una respuesta del asistente sobre un KPI?,citar fuente; fórmula; periodo; canal
9,¿Qué tier tiene derecho a una excepción de devolución por semestre?,Oro; Platino


,question,top_doc,top_score,retrieval_keyword_coverage,answer_keyword_coverage,citation_count
0,¿Cuál es el plazo de devolución para compras online?,FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx,0.206058,1.00,1.00,4
1,¿Qué productos no admiten devolución voluntaria?,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,0.228698,1.00,0.00,4
2,¿Cómo se reembolsa un pago mixto tarjeta + efectivo?,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,0.269394,1.00,0.50,4
3,¿Qué significa ventas netas y cómo se calcula?,FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.docx,0.170469,0.00,0.00,4
4,¿Qué es stock disponible?,FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.docx,0.308884,1.00,1.00,4
5,¿Qué plazo tengo para reportar un daño en transporte?,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,0.116627,1.00,1.00,4
6,¿Qué hago si el cliente no tiene ticket?,FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx,0.289226,0.50,0.00,4
7,¿En qué casos se debe escalar una devolución por sospecha de fraude?,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,0.186564,0.50,0.50,4
8,¿Qué información debe incluir una respuesta del asistente sobre un KPI?,FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.docx,0.213358,0.75,0.75,4
9,¿Qué tier tiene derecho a una excepción de devolución por semestre?,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,0.168116,1.00,1.00,4


,media
retrieval_keyword_coverage,0.775
answer_keyword_coverage,0.575
citation_count,4.000


## 5. Diagnóstico de fallos

            Un caso puede recuperar bien pero responder mal. Eso separa fallo de retrieval y fallo de generación.


In [6]:
weak_cases = rag_eval.sort_values(["answer_keyword_coverage", "retrieval_keyword_coverage"]).head(5)
display(weak_cases[["question", "expected", "top_doc", "retrieval_keyword_coverage", "answer_keyword_coverage", "answer"]])


,question,expected,top_doc,retrieval_keyword_coverage,answer_keyword_coverage,answer
3,¿Qué significa ventas netas y cómo se calcula?,Ventas brutas - descuentos - descuento por puntos - devoluciones,FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.docx,0.0,0.0,"Q: ¿Cómo se calcula 'ventas netas'? Ventas netas | Ventas brutas menos descuentos comerciales, menos descuento por puntos, menos devoluciones. Margen bruto | Ventas netas menos..."
6,¿Qué hago si el cliente no tiene ticket?,buscar por fidelización; aprobación del Store Manager,FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx,0.5,0.0,Q: ¿Qué hago si el cliente no tiene ticket? Título | FAQ interna: operaciones y atención al cliente 1) El cliente solicita devolución (portal o Atención al cliente) indicando n...
1,¿Qué productos no admiten devolución voluntaria?,a medida; OUTLET; tarjetas regalo; servicios,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,1.0,0.0,Excepciones (no admiten devolución voluntaria) Devolución voluntaria | El cliente decide devolver sin defecto (p. Canal de compra | Plazo devolución voluntaria | Observaciones ...
7,¿En qué casos se debe escalar una devolución por sospecha de fraude?,patrones anómalos; prevención de pérdidas,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,0.5,0.5,Sospecha de fraude (patrones de devolución anómalos): escalar a Prevención de pérdidas con evidencia. Sospecha de fraude: no acusar al cliente; seguir protocolo de Prevención d...
2,¿Cómo se reembolsa un pago mixto tarjeta + efectivo?,primero a tarjeta; resto en efectivo,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,1.0,0.5,Pago mixto (tarjeta + efectivo) | Primero se reembolsa a tarjeta hasta el importe pagado con tarjeta; el resto en efectivo. | Pagó 30 tarjeta + 20 efectivo -> devolver 30 tarje...


## 6. Comparar una respuesta mala y una respuesta explicable

            Esta comparación sirve para clase: una respuesta fluida sin fuentes no es equivalente a una respuesta trazable y con límites.


In [7]:
query = "¿Qué hago si el cliente no tiene ticket?"
retrieved = retrieve(rag_index, query, top_k=4)
good = grounded_answer(query, retrieved)
context = "\n".join(retrieved["text"].tolist())

bad_answer = "Puedes aceptar la devolución si el cliente insiste y parece de confianza."
good_answer = good["answer"] + " Si la evidencia no es suficiente, debe escalarse al Store Manager."

comparison = pd.DataFrame([
    {"tipo": "mala respuesta", "respuesta": bad_answer, **simple_answer_rubric(bad_answer, context, "buscar por fidelización; aprobación del Store Manager")},
    {"tipo": "respuesta explicable", "respuesta": good_answer, **simple_answer_rubric(good_answer, context, "buscar por fidelización; aprobación del Store Manager")},
])
display(comparison)


,tipo,respuesta,correctness_keyword_coverage,groundedness_token_overlap,has_caveat_or_limit,length_ok
0,mala respuesta,Puedes aceptar la devolución si el cliente insiste y parece de confianza.,0.0,0.285714,0.0,1.0
1,respuesta explicable,Q: ¿Qué hago si el cliente no tiene ticket? Título | FAQ interna: operaciones y atención al cliente 1) El cliente solicita devolución (portal o Atención al cliente) indicando n...,0.0,0.880000,1.0,1.0


## 7. Guardar resultados de evaluación

            El resultado queda como evidencia de evaluación offline. En un proyecto real se compararía entre versiones de prompt, índice, modelo o chunking.


In [8]:
out_path = ARTIFACT_DIR / "demo_02_rag_eval.parquet"
rag_eval.to_parquet(out_path, index=False)
print("Evaluación guardada:", out_path)


Evaluación guardada: C:\repos\frasohome-demos\escenarios\09-genai-interpretabilidad\artifacts\demo_02_rag_eval.parquet
